### NB1 — Dataset curation and external validation preparation
### Author: Hamid Bouseber
### This notebook prepares curated COX-2 train/test and external datasets
### for the QSAR-XAI workflow.

In [29]:
# ============================================================
# NB1 — COX-2 Dataset Curation from Raw Files
# ============================================================

import os
import math
from typing import Any, Optional

import pandas as pd
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import SaltRemover

In [30]:
# ============================================================
# Configuration
# ============================================================

OUTDIR = "./data"
os.makedirs(OUTDIR, exist_ok=True)

RAW_TRAIN_TEST_PATH = f"{OUTDIR}/dataset_raw_train_test.csv"
RAW_EXTERNAL_PATH = f"{OUTDIR}/dataset_raw_external.csv"

TRAIN_TEST_MOL_PATH = f"{OUTDIR}/dataset_train_test_molecule.csv"
EXTERNAL_MOL_PATH = f"{OUTDIR}/dataset_external_molecule.csv"

MIN_ASSAY_SIZE = 20

In [31]:
# ============================================================
# Helper functions
# ============================================================

def safe_float(x: Any) -> Optional[float]:
    try:
        if x is None or pd.isna(x):
            return None
        v = float(x)
        return v if math.isfinite(v) else None
    except Exception:
        return None


def normalize_units(u: Any) -> Optional[str]:
    if u is None or pd.isna(u):
        return None
    s = str(u).strip().replace("µ", "u").replace("μ", "u")
    return s if s else None


def to_nM(value: float, units: str) -> Optional[float]:
    factors = {"M": 1e9, "mM": 1e6, "uM": 1e3, "nM": 1.0, "pM": 1e-3}
    return value * factors[units] if units in factors else None


def pic50_from_nM(ic50_nM: float) -> Optional[float]:
    if ic50_nM is None or ic50_nM <= 0:
        return None
    return 9.0 - math.log10(ic50_nM)


def standardize_smiles(smiles: Any, remover: SaltRemover.SaltRemover) -> Optional[str]:
    if smiles is None or pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None:
        return None

    mol = remover.StripMol(mol, dontRemoveEverything=True)
    frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)

    if not frags:
        return None

    mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

In [32]:
# ============================================================
# Measurement-level curation
# ============================================================

def curate_measurements(df_raw: pd.DataFrame, label: str) -> pd.DataFrame:
    df = df_raw.copy()

    df["standard_value_float"] = df["standard_value"].map(safe_float)
    df["standard_units_norm"] = df["standard_units"].map(normalize_units)

    df = df[df["standard_value_float"].notna()]
    df = df[df["standard_value_float"] > 0]

    df["ic50_nM"] = [
        to_nM(v, u) for v, u in zip(df["standard_value_float"], df["standard_units_norm"])
    ]

    df = df[df["ic50_nM"].notna()]
    df = df[df["ic50_nM"] > 0]

    remover = SaltRemover.SaltRemover()

    df["canonical_smiles_std"] = [
        standardize_smiles(smi, remover)
        for smi in tqdm(df["canonical_smiles"], desc=f"Standardizing SMILES ({label})")
    ]

    df = df[df["canonical_smiles_std"].notna()].copy()

    df["pIC50"] = df["ic50_nM"].map(pic50_from_nM)
    df = df[df["pIC50"].notna()].copy()

    print(f"\n{label} clean measurement-level:", df.shape)
    print(f"{label} unique molecules:", df["canonical_smiles_std"].nunique())
    print(
        f"{label} pIC50 range:",
        round(df["pIC50"].min(), 3),
        "to",
        round(df["pIC50"].max(), 3),
    )

    return df

In [33]:
# ============================================================
# Molecule-level aggregation
# ============================================================
def aggregate_to_molecule(df: pd.DataFrame) -> pd.DataFrame:
    agg_dict = {
        "ic50_nM": ("ic50_nM", "median"),
        "pIC50": ("pIC50", "median"),
        "n_records": ("pIC50", "size"),
        "pIC50_std": ("pIC50", "std"),
    }

    if "molecule_chembl_id" in df.columns:
        agg_dict["molecule_chembl_id"] = ("molecule_chembl_id", "first")

    if "document_chembl_id" in df.columns:
        agg_dict["document_chembl_id"] = (
            "document_chembl_id",
            lambda x: ";".join(sorted(set(x.dropna())))
        )

    if "assay_chembl_id" in df.columns:
        agg_dict["assay_chembl_id"] = (
            "assay_chembl_id",
            lambda x: ";".join(sorted(set(x.dropna())))
        )

    if "bao_label" in df.columns:
        agg_dict["bao_label"] = (
            "bao_label",
            lambda x: x.mode()[0] if not x.mode().empty else None
        )

    return (
        df.groupby("canonical_smiles_std", as_index=False)
          .agg(**agg_dict)
          .rename(columns={"canonical_smiles_std": "canonical_smiles"})
    )

In [41]:
# ============================================================
# Molecule-level aggregation
# ============================================================

def aggregate_to_molecule(df: pd.DataFrame) -> pd.DataFrame:
    agg_dict = {
        "ic50_nM": ("ic50_nM", "median"),
        "pIC50": ("pIC50", "median"),
        "n_records": ("pIC50", "size"),
        "pIC50_std": ("pIC50", "std"),
    }

    if "molecule_chembl_id" in df.columns:
        agg_dict["molecule_chembl_id"] = ("molecule_chembl_id", "first")

    if "document_chembl_id" in df.columns:
        agg_dict["document_chembl_id"] = (
            "document_chembl_id",
            lambda x: ";".join(sorted(set(x.dropna())))
        )

    if "assay_chembl_id" in df.columns:
        agg_dict["assay_chembl_id"] = (
            "assay_chembl_id",
            lambda x: ";".join(sorted(set(x.dropna())))
        )

    if "bao_label" in df.columns:
        agg_dict["bao_label"] = (
            "bao_label",
            lambda x: x.mode()[0] if not x.mode().empty else None
        )

    return (
        df.groupby("canonical_smiles_std", as_index=False)
          .agg(**agg_dict)
          .rename(columns={"canonical_smiles_std": "canonical_smiles"})
    )

In [42]:
# ============================================================
# Load prepared raw datasets
# ============================================================

df_raw_train_test = pd.read_csv(RAW_TRAIN_TEST_PATH)
df_raw_external = pd.read_csv(RAW_EXTERNAL_PATH)

print("Train/test raw:", df_raw_train_test.shape)
print("External raw:", df_raw_external.shape)

Train/test raw: (6243, 17)
External raw: (62, 3)


In [43]:
# ============================================================
# Curate train/test and external raw datasets separately
# ============================================================

df_train_test_meas = curate_measurements(df_raw_train_test, "Train/test")
df_external_meas = curate_measurements(df_raw_external, "External")

Standardizing SMILES (Train/test): 100%|███████████████████████| 6215/6215 [00:03<00:00, 1593.68it/s]



Train/test clean measurement-level: (6214, 22)
Train/test unique molecules: 4399
Train/test pIC50 range: 0.079 to 11.222


Standardizing SMILES (External): 100%|█████████████████████████████| 62/62 [00:00<00:00, 1188.59it/s]


External clean measurement-level: (62, 8)
External unique molecules: 62
External pIC50 range: 5.067 to 7.328


In [44]:
# ============================================================
# Assay filtering for train/test data
# ============================================================

assay_counts = df_train_test_meas["assay_chembl_id"].value_counts()
valid_assays = assay_counts[assay_counts >= MIN_ASSAY_SIZE].index

df_train_test_meas = df_train_test_meas[
    df_train_test_meas["assay_chembl_id"].isin(valid_assays)
].copy()

print("Minimum assay size:", MIN_ASSAY_SIZE)
print("Retained assays:", len(valid_assays))
print("Train/test measurements after assay filtering:", df_train_test_meas.shape)
print("Train/test unique molecules:", df_train_test_meas["canonical_smiles_std"].nunique())

Minimum assay size: 20
Retained assays: 82
Train/test measurements after assay filtering: (3189, 22)
Train/test unique molecules: 2297


In [45]:
# ============================================================
# Remove molecular overlap between train/test and external
# ============================================================

train_smiles = set(df_train_test_meas["canonical_smiles_std"])

df_external_meas = df_external_meas[
    ~df_external_meas["canonical_smiles_std"].isin(train_smiles)
].copy()

print("External measurements after overlap removal:", df_external_meas.shape)
print("External unique molecules after overlap removal:", df_external_meas["canonical_smiles_std"].nunique())

External measurements after overlap removal: (58, 8)
External unique molecules after overlap removal: 58


In [46]:
# ============================================================
# Aggregate to molecule level
# ============================================================

df_train_test_mol = aggregate_to_molecule(df_train_test_meas)
df_external_mol = aggregate_to_molecule(df_external_meas)

print("Train/test molecule-level:", df_train_test_mol.shape)
print("External molecule-level:", df_external_mol.shape)

print("\nTrain/test pIC50:")
print(df_train_test_mol["pIC50"].describe().round(3))

print("\nExternal pIC50:")
print(df_external_mol["pIC50"].describe().round(3))

Train/test molecule-level: (2297, 9)
External molecule-level: (58, 5)

Train/test pIC50:
count    2297.000
mean        6.322
std         1.259
min         1.222
25%         5.495
50%         6.456
75%         7.208
max        11.222
Name: pIC50, dtype: float64

External pIC50:
count    58.000
mean      6.168
std       0.600
min       5.067
25%       5.691
50%       6.284
75%       6.653
max       7.328
Name: pIC50, dtype: float64


In [47]:
# ============================================================
# Final independence checks
# ============================================================

overlap = set(df_train_test_mol["canonical_smiles"]).intersection(
    set(df_external_mol["canonical_smiles"])
)

print("Molecular overlap:", len(overlap))

assert len(overlap) == 0, "Overlap detected between train/test and external datasets."
assert len(df_external_mol) > 0, "External dataset is empty."
assert df_train_test_mol["pIC50"].notna().all()
assert df_external_mol["pIC50"].notna().all()

Molecular overlap: 0


In [48]:
# ============================================================
# Save full molecule-level datasets for pipeline
# ============================================================

df_train_test_mol.to_csv(TRAIN_TEST_MOL_PATH, index=False)
df_external_mol.to_csv(EXTERNAL_MOL_PATH, index=False)

print("Saved full pipeline datasets:")
print(TRAIN_TEST_MOL_PATH)
print(EXTERNAL_MOL_PATH)

Saved full pipeline datasets:
./data/dataset_train_test_molecule.csv
./data/dataset_external_molecule.csv
